# Customer Segmentation Analysis
## Oasis Infobyte — Data Analytics Task 2

RFM analysis and K-Means clustering for e-commerce customer segmentation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
df = pd.read_csv("data/marketing_campaign.csv", sep="\t")
df.head()

## 1. Dataset Inspection

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nMissing values:")
print(df.isna().sum()[df.isna().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())
df.info()

## 2. Data Cleaning

In [ ]:
df = df.drop_duplicates().copy()
df["Income"] = df["Income"].fillna(df["Income"].median())
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], dayfirst=True, errors="coerce")
df = df[df["Year_Birth"].between(1900, 2000)].copy()
df = df[df["Income"] >= 0].copy()
print("Cleaned shape:", df.shape)

## 3. RFM / Behavioural Feature Engineering

In [ ]:
spend_cols = [
    "MntWines", "MntFruits", "MntMeatProducts",
    "MntFishProducts", "MntSweetProducts", "MntGoldProds"
]
purchase_cols = [
    "NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases"
]

df["Monetary"] = df[spend_cols].sum(axis=1)
df["Frequency"] = df[purchase_cols].sum(axis=1)
df["AveragePurchaseValue"] = np.where(
    df["Frequency"] > 0,
    df["Monetary"] / df["Frequency"],
    0
)
df["CLV_Proxy"] = df["Monetary"]

rfm = df[["Recency", "Frequency", "Monetary", "AveragePurchaseValue", "CLV_Proxy"]]
rfm.describe().T.round(2)

## 4. Standardisation

In [ ]:
model_df = df[["Recency", "Frequency", "Monetary"]].copy()
model_df["Frequency"] = np.log1p(model_df["Frequency"])
model_df["Monetary"] = np.log1p(model_df["Monetary"])

scaler = StandardScaler()
X = scaler.fit_transform(model_df)
X[:5]

## 5. Elbow Method

In [ ]:
k_values = range(2, 9)
inertias = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(list(k_values), inertias, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.grid(alpha=0.25)
plt.show()

## 6. K-Means Clustering

In [ ]:
K = 4
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X)

silhouette = silhouette_score(X, df["Cluster"])
print("Silhouette Score:", round(silhouette, 3))

## 7. Cluster Visualisation

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df, x="Frequency", y="Monetary",
    hue="Cluster", palette="tab10", s=55, alpha=0.75
)
plt.title("Frequency vs Monetary")
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df, x="Recency", y="Monetary",
    hue="Cluster", palette="tab10", s=55, alpha=0.75
)
plt.title("Recency vs Monetary")
plt.show()

## 8. Cluster Profiling

In [ ]:
profile = (
    df.groupby("Cluster")
      .agg(
          Customers=("ID", "count"),
          Recency=("Recency", "mean"),
          Frequency=("Frequency", "mean"),
          Monetary=("Monetary", "mean"),
          AveragePurchaseValue=("AveragePurchaseValue", "mean"),
          CLV_Proxy=("CLV_Proxy", "mean")
      )
      .round(2)
)
profile

In [ ]:
profile["Customers"].plot(kind="bar", figsize=(8, 4), title="Customers per Cluster")
plt.xlabel("Cluster")
plt.ylabel("Customers")
plt.show()

## 9. Business Insights & Marketing Recommendations

Interpret clusters using Recency, Frequency and Monetary values. Typical actions include VIP rewards for high-value loyal customers, win-back campaigns for high-value customers with high recency, reactivation for low-engagement customers, and cross-sell/upsell campaigns for recent big spenders.

**CLV limitation:** this customer-level dataset does not contain transaction-level history, lifespan or margin, so historical Monetary spend is used as a CLV proxy rather than exact CLV.